# Tc Human Annotation Visualisation

Comparison of pipeline-extracted critical temperature (Tc) values against human annotations.

In [ ]:
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

# Project style
SRC_DIR = str(Path().resolve().parents[2] / "src")
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)
from llm_synthesis.utils.style_utils import set_style, get_palette
set_style("manuscript")

plt.rcParams.update({
    "figure.dpi": 150,
    "savefig.dpi": 300,
    "savefig.bbox": "tight",
    "axes.grid": False,
})
# Force a safe built-in math font so $T_c$ etc. never trigger a missing-font error
import matplotlib
matplotlib.rcParams["mathtext.fontset"] = "stixsans"
matplotlib.rcParams["mathtext.default"] = "regular"

sns.set_style("white")

PAL = get_palette()
C_BLUE    = PAL[2]
C_ORANGE  = PAL[4]
C_PURPLE  = PAL[12]
C_PINK    = PAL[0]
C_LBLUE   = PAL[3]
C_LPURPLE = PAL[13]
C_GREY    = PAL[8]
C_DGREY   = PAL[10]
C_BLACK   = PAL[11]
C_LAVENDER = PAL[1]

print("Style loaded")

## Data Loading & Preparation

In [ ]:
                                                                                                              
#DATA_PATH = Path.home() / "Downloads" / "tc_master_snippet_human_annotation.xlsx"
DATA_PATH = Path.home() / "Documents" / "Studieren" / "lemat_synth" / "lematerial-llm-synthesis" / "examples" / "data" / "results_Tc_v3_24_02" / "tc_master_snippet_v3_annotated_v2.csv"

if DATA_PATH.suffix == ".xlsx":
    df = pd.read_excel(DATA_PATH)
else:
    df = pd.read_csv(DATA_PATH)
print(f"Loaded: {len(df)} rows, {df.columns.size} columns")

# --- Harmonise column names (works for both xlsx and merged v3 csv) ---
rename_map = {
    "Tc_text human ": "tc_text_human",
    "Tc_human":       "tc_human",
    "Human: comment ":"human_comment",
}
df.rename(columns={k: v for k, v in rename_map.items() if k in df.columns}, inplace=True)

# --- Drop rows where human annotation is absent (string "NA" from merge, or actual NaN) ---
df = df[df["tc_text_human"].notna() & (df["tc_text_human"].astype(str) != "NA")]
df = df[df["tc_human"].notna() & (df["tc_human"].astype(str) != "NA")]
print(f"After filtering for human annotations: {len(df)} rows")

# --- Parse human annotation columns ---
# "Not written" = human confirmed no Tc in text for this material
# "Not in plot" = human confirmed material is not in any figure
# "Not SC"      = human confirmed material is NOT superconducting
# Numeric string = human-annotated Tc value
# NaN = not annotated

# Keep raw strings for FP/FN logic, create numeric versions for plotting
df["Tc_text_human_raw"] = df["tc_text_human"].astype(str).str.strip()
df["Tc_human_raw"] = df["tc_human"].astype(str).str.strip()

df["Tc_text_human"] = pd.to_numeric(df["tc_text_human"], errors="coerce")
df["Tc_human_num"] = pd.to_numeric(df["tc_human"], errors="coerce")

# Boolean flags: human explicitly says "not there" or "not SC"
df["text_human_not_written"] = df["Tc_text_human_raw"].str.lower().str.contains("not written", na=False)
df["plot_human_not_in_plot"] = df["Tc_human_raw"].str.lower().str.contains("not in plot", na=False)
df["human_not_sc"] = df["Tc_human_raw"].str.lower().str.contains("not sc", na=False)

# Create tc_text_first: first non-null of tc_text, tc_text_onset, tc_text_zero
df["tc_text_first"] = df["tc_text"].fillna(df["tc_text_onset"]).fillna(df["tc_text_zero"])

print(f"\nNon-null counts for key columns:")
for col in ["tc_text_first", "Tc_text_human", "Tc_human_num",
            "tc_vlm_orig", "tc_vlm_orig_onset", "tc_vlm_orig_zero",
            "tc_vlm_snip", "tc_vlm_snip_onset", "tc_vlm_snip_zero"]:
    print(f"  {col:25s}: {df[col].notna().sum()}")

print(f"\nHuman 'Not written' count: {df['text_human_not_written'].sum()}")
print(f"Human 'Not in plot' count: {df['plot_human_not_in_plot'].sum()}")
print(f"Human 'Not SC' count:      {df['human_not_sc'].sum()}")

df.head(3)

## Helper Function

In [ ]:
def parity_plot(x, y, xlabel, ylabel, title, ax,
                color=None, fp_mask=None, fn_mask=None, fp_vals=None, fn_vals=None):
    """
    Parity plot with identity line, regression, error metrics,
    and marginal markers for false positives / false negatives.

    Parameters
    ----------
    x, y : pd.Series — pipeline/predicted vs human/reference values (numeric, NaN = missing)
    fp_mask : bool Series — True where pipeline extracted a value but human says "not there"
    fn_mask : bool Series — True where human has a value but pipeline has NaN
    fp_vals : pd.Series — the pipeline values for FP points (plotted along x-axis)
    fn_vals : pd.Series — the human values for FN points (plotted along y-axis)
    """
    if color is None:
        color = C_BLUE

    # True positives: both have numeric values
    tp_mask = x.notna() & y.notna()
    x_tp, y_tp = x[tp_mask].values, y[tp_mask].values
    n_tp = len(x_tp)

    # Count FP / FN
    n_fp = fp_mask.sum() if fp_mask is not None else 0
    n_fn = fn_mask.sum() if fn_mask is not None else 0

    if n_tp < 2:
        ax.text(0.5, 0.5, f"Not enough matched data (n={n_tp})",
                transform=ax.transAxes, ha="center", va="center", fontsize=10)
        ax.set_title(title)
        return

    max_val = max(x_tp.max(), y_tp.max()) * 1.15
    min_val = min(x_tp.min(), y_tp.min())
    pad = (max_val - min_val) * 0.05

    # Identity line
    ax.plot([min_val - pad, max_val], [min_val - pad, max_val],
            "k--", lw=1, alpha=0.5, label="y = x")

    # Regression + scatter for true positives
    sns.regplot(x=x_tp, y=y_tp, ci=95, ax=ax,
                scatter_kws={"s": 45, "edgecolors": "k", "linewidths": 0.3,
                             "alpha": 0.8, "zorder": 3, "color": color},
                line_kws={"lw": 1.5, "color": C_PURPLE})

    # False positives: pipeline extracted but human says not there → along x-axis
    if fp_vals is not None and n_fp > 0:
        y_fp_pos = min_val - pad * 1.5
        ax.scatter(fp_vals[fp_mask].values, [y_fp_pos] * n_fp,
                   c=C_ORANGE, marker="v", s=30, alpha=0.8, edgecolors="k",
                   linewidths=0.3, zorder=4, label=f"FP (n={n_fp})")

    # False negatives: human has value but pipeline missed → along y-axis
    if fn_vals is not None and n_fn > 0:
        x_fn_pos = min_val - pad * 1.5
        ax.scatter([x_fn_pos] * n_fn, fn_vals[fn_mask].values,
                   c=C_GREY, marker="<", s=30, alpha=0.8, edgecolors="k",
                   linewidths=0.3, zorder=4, label=f"FN (n={n_fn})")

    # Metrics
    r2 = r2_score(y_tp, x_tp)
    mae = mean_absolute_error(y_tp, x_tp)
    rmse = np.sqrt(mean_squared_error(y_tp, x_tp))
    # MAPE: mean absolute percentage error (relative to reference y)
    nonzero = y_tp != 0
    if nonzero.sum() > 0:
        mape = np.mean(np.abs((x_tp[nonzero] - y_tp[nonzero]) / y_tp[nonzero])) * 100
    else:
        mape = np.nan

    r2_str = f"{r2:.3f}" if np.isfinite(r2) else "N/A"
    mae_str = f"{mae:.1f}" if np.isfinite(mae) else "N/A"
    rmse_str = f"{rmse:.1f}" if np.isfinite(rmse) else "N/A"
    mape_str = f"{mape:.1f}%" if np.isfinite(mape) else "N/A"

    stats_text = (f"R\u00b2 = {r2_str}\nMAE = {mae_str} K\n"
                  f"RMSE = {rmse_str} K\nMAPE = {mape_str}\n"
                  f"TP = {n_tp}")
    if n_fp > 0:
        stats_text += f"  FP = {n_fp}"
    if n_fn > 0:
        stats_text += f"  FN = {n_fn}"

    ax.text(0.05, 0.95, stats_text,
            transform=ax.transAxes, fontsize=8, verticalalignment="top",
            bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.8))

    lim_lo = min_val - pad * 3 if (n_fp > 0 or n_fn > 0) else min_val - pad
    ax.set_xlim(lim_lo, max_val)
    ax.set_ylim(lim_lo, max_val)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.set_aspect("equal")
    ax.legend(fontsize=7, loc="lower right")

---
## 1 · Pipeline Text Extraction vs Human Text Annotation

`tc_text_first` (first non-null of tc_text / tc_text_onset / tc_text_zero) vs `Tc_text human`

- **FP**: pipeline extracted a Tc from text, but human says "Not written" → false detection
- **FN**: human found a Tc in text, but pipeline returned NaN → missed extraction

In [ ]:
# FP: pipeline extracted text Tc but human says "Not written"
fp_text = df["tc_text_first"].notna() & df["text_human_not_written"]
# FN: human has a text Tc value but pipeline missed it
fn_text = df["tc_text_first"].isna() & df["Tc_text_human"].notna()

fig, ax = plt.subplots(figsize=(7, 6))
parity_plot(
    df["tc_text_first"], df["Tc_text_human"],
    xlabel="Tc pipeline text (K)",
    ylabel="Tc human text (K)",
    title="Pipeline Text vs Human Text Annotation",
    ax=ax, color=C_BLUE,
    fp_mask=fp_text, fp_vals=df["tc_text_first"],
    fn_mask=fn_text, fn_vals=df["Tc_text_human"],
)

ax.set_xlabel(ax.get_xlabel(), fontsize=16)
ax.set_ylabel(ax.get_ylabel(), fontsize=16)
ax.set_title(ax.get_title(), fontsize=18)
ax.tick_params(axis="both", labelsize=14)
if ax.get_legend():
    ax.legend(fontsize=12)
if ax.texts:
    ax.texts[0].set_fontsize(14)

plt.tight_layout()
plt.show()

n_tp = (df["tc_text_first"].notna() & df["Tc_text_human"].notna()).sum()
print(f"\n--- Pipeline Text vs Human Text ---")
print(f"TP ({n_tp}): Both pipeline and human found a Tc value in the text.")
print(f"FP ({fp_text.sum()}): Pipeline extracted a Tc from text, but human says 'Not written' — false detection / hallucination.")
print(f"FN ({fn_text.sum()}): Human found a Tc in the text, but pipeline returned nothing — missed extraction.")


---
## 2 · Human Ground-Truth Tc vs Each VLM Extraction

`Tc_human` (from plot) compared separately against each of the six VLM columns.

- **FP**: VLM extracted a value but human says "Not in plot" → false detection
- **FN**: human has a plot Tc but VLM returned NaN → missed extraction

In [ ]:
vlm_cols = [
    ("tc_vlm_orig",       "VLM orig"),
    ("tc_vlm_orig_onset", "VLM orig onset"),
    ("tc_vlm_orig_zero",  "VLM orig zero"),
    ("tc_vlm_snip",       "VLM snip"),
    ("tc_vlm_snip_onset", "VLM snip onset"),
    ("tc_vlm_snip_zero",  "VLM snip zero"),
]

vlm_colors = [C_BLUE, C_LBLUE, C_LAVENDER, C_PINK, C_LPURPLE, C_ORANGE]

fig, axes = plt.subplots(2, 3, figsize=(14, 9))

for ax, (col, label), color in zip(axes.flat, vlm_cols, vlm_colors):
    # FP: VLM extracted but human says "Not in plot"
    fp = df[col].notna() & df["plot_human_not_in_plot"]
    # FN: human has plot Tc but VLM missed
    fn = df[col].isna() & df["Tc_human_num"].notna()

    parity_plot(
        df[col], df["Tc_human_num"],
        xlabel=f"Tc {label} (K)",
        ylabel="Tc human (K) VLM",
        title=f"{label} vs Human Tc",
        ax=ax, color=color,
        fp_mask=fp, fp_vals=df[col],
        fn_mask=fn, fn_vals=df["Tc_human_num"],
    )

plt.tight_layout()
plt.show()

print("\n--- Tc_human (from plot) vs each VLM extraction ---")
print("For each subplot:")
print("  TP: Both VLM and human found a Tc value from the figure.")
print("  FP: VLM extracted a Tc from the figure, but human says 'Not in plot' — false detection.")
print("  FN: Human read a Tc from the figure, but VLM returned nothing — missed extraction.\n")
for col, label in vlm_cols:
    fp = df[col].notna() & df["plot_human_not_in_plot"]
    fn = df[col].isna() & df["Tc_human_num"].notna()
    tp = (df[col].notna() & df["Tc_human_num"].notna()).sum()
    print(f"  {label:20s}  TP={tp:3d}   FP={fp.sum():3d}   FN={fn.sum():3d}")

---
## 2b · VLM Figure Extraction vs Human Text Annotation (cross-modal)

Can the VLM figure reader recover the same Tc the human found in the text?

| Category | Marker | Meaning |
|---|---|---|
| **TP** | coloured dots on parity | Both VLM (from figure) and human (from text) have a numeric Tc |
| **Figure-only Tc** | green ▲ along x-axis | VLM found Tc from the figure, but human says it's *not written* in the text → **VLM adds value!** |
| **Not SC → FP** | red ▼ along x-axis | VLM reports a Tc, but human says this material is *not superconducting* → **real false positive** |
| **Missed by VLM** | grey ◁ along y-axis | Human found Tc in the text, but VLM returned nothing from figures → missed extraction |

In [ ]:
#plot 2b — VLM extraction (x) vs Human TEXT annotation (y)
# Cross-modal comparison: can the VLM figure reader recover the same Tc the human found in text?
#
# Four visual categories:
#   TP (blue dots)      → Both VLM and human text agree on a numeric Tc
#   "Figure-only Tc"    → Tc not written in text, VLM found it from figure = VLM adds value (green ▲)
#   "Not SC → FP"       → Human says NOT superconducting, VLM still reports Tc = hallucination (red ▼)
#   "Missed by VLM"     → Human found Tc in text, VLM returned nothing (grey ◁)

vlm_cols = [
    ("tc_vlm_orig",       "VLM orig"),
    ("tc_vlm_orig_onset", "VLM orig onset"),
    ("tc_vlm_orig_zero",  "VLM orig zero"),
    ("tc_vlm_snip",       "VLM snip"),
    ("tc_vlm_snip_onset", "VLM snip onset"),
    ("tc_vlm_snip_zero",  "VLM snip zero"),
]
vlm_colors = [C_BLUE, C_LBLUE, C_LAVENDER, C_PINK, C_LPURPLE, C_ORANGE]

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
fig.suptitle(
    "VLM figure extraction vs Human text annotation\n"
    "(cross-modal: does the VLM recover Tc that humans found in the text?)",
    fontsize=12, y=1.02,
)

for ax, (col, label), color in zip(axes.flat, vlm_cols, vlm_colors):
    # ── Define categories ──
    tp   = df[col].notna() & df["Tc_text_human"].notna()
    # VLM found Tc from figure, human says not written in text, but material IS SC
    fig_only = df[col].notna() & df["text_human_not_written"] & ~df["human_not_sc"]
    # VLM reports Tc but human says material is NOT superconducting at all
    not_sc   = df[col].notna() & df["human_not_sc"]
    # Human found Tc in text but VLM returned nothing from the figure
    missed   = df[col].isna() & df["Tc_text_human"].notna()

    # ── Parity plot for TP only (we add our own marginal markers) ──
    parity_plot(
        df[col], df["Tc_text_human"],
        xlabel=f"Tc {label} (K)",
        ylabel="Tc human text (K)",
        title=label,
        ax=ax, color=color,
        fp_mask=None, fp_vals=None,
        fn_mask=None, fn_vals=None,   # we draw FN ourselves for consistent labelling
    )

    # ── Axis limits for margin placement ──
    xlim = ax.get_xlim()
    ylim = ax.get_ylim()
    y_range = ylim[1] - ylim[0]
    x_range = xlim[1] - xlim[0]

    # ── GREEN ▲: Figure-only Tc  (VLM wins — found Tc not in text) ──
    n_fig = fig_only.sum()
    if n_fig > 0:
        margin_y1 = ylim[0] + y_range * 0.02
        ax.scatter(
            df.loc[fig_only, col].values,
            [margin_y1] * n_fig,
            c="seagreen", marker="^", s=40, alpha=0.85,
            edgecolors="k", linewidths=0.3, zorder=5,
            label=f"Figure-only Tc  (n={n_fig})",
        )

    # ── RED ▼: Not SC → false positive (VLM hallucinated Tc) ──
    n_notsc = not_sc.sum()
    if n_notsc > 0:
        margin_y2 = ylim[0] + y_range * 0.07
        ax.scatter(
            df.loc[not_sc, col].values,
            [margin_y2] * n_notsc,
            c="crimson", marker="v", s=40, alpha=0.85,
            edgecolors="k", linewidths=0.3, zorder=5,
            label=f"Not SC → FP  (n={n_notsc})",
        )

    # ── GREY ◁: Missed by VLM (human text Tc but VLM missed) ──
    n_miss = missed.sum()
    if n_miss > 0:
        margin_x = xlim[0] + x_range * 0.02
        ax.scatter(
            [margin_x] * n_miss,
            df.loc[missed, "Tc_text_human"].values,
            c="silver", marker="<", s=35, alpha=0.8,
            edgecolors="k", linewidths=0.3, zorder=4,
            label=f"Missed by VLM  (n={n_miss})",
        )

    # ── Stats box: update with all category counts ──
    n_tp = tp.sum()
    old_txt = ax.texts[0].get_text() if ax.texts else ""
    new_stats = (
        old_txt
        + f"\n─────────────────"
        + f"\nFigure-only Tc = {n_fig}"
        + f"\nNot SC (FP) = {n_notsc}"
        + f"\nMissed by VLM = {n_miss}"
    )
    if ax.texts:
        ax.texts[0].set_text(new_stats)

    ax.legend(fontsize=5.5, loc="lower right", framealpha=0.9)

plt.tight_layout()
plt.show()

# ── Summary table ──
print("\n--- VLM figure extraction (x) vs Human text annotation (y) ---")
print("Cross-modal: does the VLM figure reader recover the same Tc the human found in text?\n")
print("  TP (dots):            Both VLM and human text have a numeric Tc — agreement.")
print("  Figure-only Tc (▲):   VLM found Tc from figure, human says NOT in text — VLM adds value!")
print("  Not SC → FP (▼):      VLM reports Tc, but human says NOT superconducting — hallucination.")
print("  Missed by VLM (◁):   Human found Tc in text, VLM returned nothing — missed extraction.\n")

header = f"  {'Method':<20s}  {'TP':>4s}   {'Fig-only':>8s}   {'Not SC':>6s}   {'Missed':>6s}"
print(header)
print("  " + "─" * len(header.strip()))
for col, label in vlm_cols:
    n_tp   = (df[col].notna() & df["Tc_text_human"].notna()).sum()
    n_fig  = (df[col].notna() & df["text_human_not_written"] & ~df["human_not_sc"]).sum()
    n_fp   = (df[col].notna() & df["human_not_sc"]).sum()
    n_miss = (df[col].isna() & df["Tc_text_human"].notna()).sum()
    print(f"  {label:<20s}  {n_tp:4d}   {n_fig:8d}   {n_fp:6d}   {n_miss:6d}")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))

col, label, color = "tc_vlm_orig", "VLM orig", C_BLUE

tp = df[col].notna() & df["Tc_text_human"].notna()
not_written = df[col].notna() & df["text_human_not_written"] & ~df["human_not_sc"]
real_fp = df[col].notna() & df["human_not_sc"]
fn = df[col].isna() & df["Tc_text_human"].notna()

parity_plot(
    df[col], df["Tc_text_human"],
    xlabel=f"Tc {label} (K)", ylabel="Tc human text (K)",
    title=f"{label} vs Human Text Tc",
    ax=ax, color=color,
    fp_mask=None, fp_vals=None,
    fn_mask=fn, fn_vals=df["Tc_text_human"],
)

xlim, ylim = ax.get_xlim(), ax.get_ylim()
margin_y = ylim[0] + (ylim[1] - ylim[0]) * 0.02

n_nw = not_written.sum()
if n_nw > 0:
    ax.scatter(df.loc[not_written, col].values, [margin_y] * n_nw,
               c="green", marker="^", s=50, alpha=0.85,
               edgecolors="k", linewidths=0.3, zorder=5,
               label=f"Not written (n={n_nw})")

n_fp = real_fp.sum()
if n_fp > 0:
    margin_y2 = ylim[0] + (ylim[1] - ylim[0]) * 0.06
    ax.scatter(df.loc[real_fp, col].values, [margin_y2] * n_fp,
               c="red", marker="v", s=50, alpha=0.85,
               edgecolors="k", linewidths=0.3, zorder=5,
               label=f"Not SC (n={n_fp})")

old_txt = ax.texts[0].get_text() if ax.texts else ""
if ax.texts:
    ax.texts[0].set_text(old_txt + f"\nNot written = {n_nw}\nNot SC = {n_fp}")
    ax.texts[0].set_fontsize(14)

ax.set_xlabel(ax.get_xlabel(), fontsize=16)
ax.set_ylabel(ax.get_ylabel(), fontsize=16)
ax.set_title(ax.get_title(), fontsize=18)
ax.tick_params(axis="both", labelsize=14)
ax.legend(fontsize=12, loc="lower right")

plt.tight_layout()
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))

col, label, color = "tc_vlm_snip", "VLM snip", C_PINK

tp = df[col].notna() & df["Tc_text_human"].notna()
not_written = df[col].notna() & df["text_human_not_written"] & ~df["human_not_sc"]
real_fp = df[col].notna() & df["human_not_sc"]
fn = df[col].isna() & df["Tc_text_human"].notna()

parity_plot(
    df[col], df["Tc_text_human"],
    xlabel=f"Tc {label} (K)", ylabel="Tc human text (K)",
    title=f"{label} vs Human Text Tc",
    ax=ax, color=color,
    fp_mask=None, fp_vals=None,
    fn_mask=fn, fn_vals=df["Tc_text_human"],
)

xlim, ylim = ax.get_xlim(), ax.get_ylim()
margin_y = ylim[0] + (ylim[1] - ylim[0]) * 0.02

n_nw = not_written.sum()
if n_nw > 0:
    ax.scatter(df.loc[not_written, col].values, [margin_y] * n_nw,
               c="green", marker="^", s=50, alpha=0.85,
               edgecolors="k", linewidths=0.3, zorder=5,
               label=f"Not written (n={n_nw})")

n_fp = real_fp.sum()
if n_fp > 0:
    margin_y2 = ylim[0] + (ylim[1] - ylim[0]) * 0.06
    ax.scatter(df.loc[real_fp, col].values, [margin_y2] * n_fp,
               c="red", marker="v", s=50, alpha=0.85,
               edgecolors="k", linewidths=0.3, zorder=5,
               label=f"Not SC (n={n_fp})")

old_txt = ax.texts[0].get_text() if ax.texts else ""
if ax.texts:
    ax.texts[0].set_text(old_txt + f"\nNot written = {n_nw}\nNot SC = {n_fp}")
    ax.texts[0].set_fontsize(14)

ax.set_xlabel(ax.get_xlabel(), fontsize=16)
ax.set_ylabel(ax.get_ylabel(), fontsize=16)
ax.set_title(ax.get_title(), fontsize=18)
ax.tick_params(axis="both", labelsize=14)
ax.legend(fontsize=12, loc="lower right")

plt.tight_layout()
plt.show()


---
## 3 · Human Text Annotation vs Human Ground-Truth Tc

`Tc_text human` vs `Tc_human` — which materials appear in text only, plot only, or both?

- Points on parity: material has Tc in both text and plot
- **Text only** (along x-axis): human found Tc in text but says "Not in plot"
- **Plot only** (along y-axis): human found Tc in plot but says "Not written" in text

In [ ]:
# Text only: human has text Tc but says "Not in plot"
text_only = df["Tc_text_human"].notna() & df["plot_human_not_in_plot"]
# Plot only: human has plot Tc but says "Not written" in text
plot_only = df["Tc_human_num"].notna() & df["text_human_not_written"]

fig, ax = plt.subplots(figsize=(7, 6))
parity_plot(
    df["Tc_text_human"], df["Tc_human_num"],
    xlabel="Tc human text (K)",
    ylabel="Tc human plot (K)",
    title="Human Text vs Human Plot Tc",
    ax=ax, color=C_PINK,
    fp_mask=text_only, fp_vals=df["Tc_text_human"],
    fn_mask=plot_only, fn_vals=df["Tc_human_num"],
)

# Relabel legend entries for this specific plot
handles, labels = ax.get_legend_handles_labels()
new_labels = []
for lbl in labels:
    if lbl.startswith("FP"):
        new_labels.append(lbl.replace("FP", "Text only"))
    elif lbl.startswith("FN"):
        new_labels.append(lbl.replace("FN", "Plot only"))
    else:
        new_labels.append(lbl)
ax.legend(handles, new_labels, fontsize=12, loc="lower right")

ax.set_xlabel(ax.get_xlabel(), fontsize=16)
ax.set_ylabel(ax.get_ylabel(), fontsize=16)
ax.set_title(ax.get_title(), fontsize=18)
ax.tick_params(axis="both", labelsize=14)
if ax.texts:
    ax.texts[0].set_fontsize(14)

plt.tight_layout()
plt.show()

n_both = (df["Tc_text_human"].notna() & df["Tc_human_num"].notna()).sum()
print(f"\n--- Human Text vs Human Plot ---")
print(f"Both ({n_both}): Human found a Tc in both the text and the figure for this material.")
print(f"Text only ({text_only.sum()}): Human found a Tc in the text, but says 'Not in plot' — value only reported in text.")
print(f"Plot only ({plot_only.sum()}): Human read a Tc from the figure, but says 'Not written' — value only visible in a figure.")


---
# Synthesis & Material Landscape (v3 data)

Load v3 outputs (JSON per material) which include synthesis extraction:
method, starting materials, steps with temperatures/durations, equipment, and evaluation scores.

In [ ]:
import json, re, glob
from collections import Counter

# ── paths ──
V3_DIR = Path().resolve().parents[2] / "examples" / "data" / "results_Tc_v3_24_02"
V3_CSV = V3_DIR / "tc_master_snippet_v3.csv"

# ── 1. Load flat CSV (one row per material) ──
dfv = pd.read_csv(V3_CSV)
dfv["tc_best"] = pd.to_numeric(dfv["tc_best"], errors="coerce")
print(f"v3 CSV: {len(dfv)} rows, {dfv['paper_id'].nunique()} papers, {dfv['material'].nunique()} materials")

# ── 2. Parse every material JSON for rich synthesis fields ──
records = []
for jf in sorted(V3_DIR.glob("*/*.json")):
    if jf.name in ("summary.json", "tc_flat_records.jsonl"):
        continue
    if jf.stat().st_size == 0:
        continue
    try:
        d = json.loads(jf.read_text())
    except json.JSONDecodeError:
        continue

    syn  = d.get("synthesis") or {}
    eva  = (d.get("evaluation") or {}).get("scores") or {}
    tc_t = d.get("tc_from_text") or []

    # best Tc from text block
    tc_mid = None
    for t in tc_t:
        v = t.get("Tc_mid") or t.get("T_onset") or t.get("T_zero")
        if v is not None:
            tc_mid = float(v)
            break

    steps = syn.get("steps") or []
    # extract max sintering temperature from step conditions
    temps = []
    durations = []
    for s in steps:
        c = (s.get("conditions") if isinstance(s, dict) else None) or {}
        if isinstance(c, dict):
            t = c.get("temperature")
            if t is not None:
                try: temps.append(float(t))
                except: pass
            dur = c.get("duration")
            if dur is not None:
                try: durations.append(float(dur))
                except: pass

    starting = syn.get("starting_materials") or []
    starting_names = []
    for m in starting:
        n = m.get("name", str(m)) if isinstance(m, dict) else str(m)
        starting_names.append(n)

    equip = syn.get("equipment") or []
    equip_names = [e.get("name", str(e)) if isinstance(e, dict) else str(e) for e in equip]

    records.append({
        "paper_id":         jf.parent.name,
        "material":         d.get("material", jf.stem),
        "method":           syn.get("synthesis_method") or "unknown",
        "compound_type":    syn.get("target_compound_type") or "unknown",
        "n_steps":          len(steps),
        "max_temp_C":       max(temps) if temps else np.nan,
        "total_duration_h": sum(durations) if durations else np.nan,
        "n_starting":       len(starting_names),
        "starting_materials": starting_names,
        "equipment":        equip_names,
        "overall_score":    eva.get("overall_score"),
        "tc_mid_text":      tc_mid,
    })

dfs = pd.DataFrame(records)
print(f"Parsed {len(dfs)} material JSONs with synthesis data")
print(f"\nSynthesis methods:\n{dfs['method'].value_counts().to_string()}")
print(f"\nCompound types:\n{dfs['compound_type'].value_counts().to_string()}")

# ── 3. Merge JSON synthesis details onto the flat CSV ──
dfv = dfv.merge(
    dfs[["paper_id", "material", "method", "compound_type", "n_steps",
         "max_temp_C", "total_duration_h", "n_starting", "overall_score"]],
    on=["paper_id", "material"], how="left", suffixes=("", "_json"),
)
# fill method from CSV column where JSON is missing
dfv["method"] = dfv["method"].fillna(dfv["synthesis_method"])
print(f"\nMerged: {dfv['method'].notna().sum()}/{len(dfv)} rows with a synthesis method")

In [ ]:
# ── Material family classifier ──
# Rule-based: covers the families in this dataset and extends to common SC families

def classify_family(name: str) -> str:
    """Classify a material name into a superconductor family."""
    s = str(name)
    # Iron chalcogenides  (FeTeSe, FeSe, ...)
    if re.search(r"Fe.*Te|Fe.*Se|FeSe|FeTe", s, re.I):
        return "Fe-chalcogenide"
    # Iron pnictides  (LaFeAsO, CaFeAsF, NdOFeAs, BaFe2As2, SrFeAsF, ...)
    if re.search(r"Fe.*As|FeAs|Fe2As2", s, re.I):
        return "Fe-pnictide"
    # Cuprates  (La-Ba-Cu-O, Bi-Sr-Cu-O, Ru-Sr-Gd-Cu-O, (Cu,C)Ba2Ca2Cu3O9, ...)
    if re.search(r"Cu.*O|CuO", s, re.I):
        return "Cuprate"
    # MgB2 family  (MgB2, Mg-Al-B, ...)
    if re.search(r"Mg.*B|MgB", s, re.I):
        return "MgB2-type"
    # Re-Mo alloys
    if re.search(r"Re.*Mo|Mo.*Re", s, re.I):
        return "Re-Mo alloy"
    # Transition-metal pnictides / phosphides  (WP, CrAs, MnP, FeP, ...)
    if re.search(r"^(W|Cr|Mn|Co|Ir|Ru|Rh|Fe|Ni)(P|As)$", s):
        return "TM mono-pnictide"
    if "RuP" in s or "Rh-doped" in s:
        return "TM mono-pnictide"
    # Ta-Pd-S
    if re.search(r"Ta.*Pd.*S", s, re.I):
        return "Ta-Pd-S"
    # Pd-HoTe3
    if re.search(r"Pd.*HoTe|HoTe", s, re.I):
        return "Pd-HoTe3"
    # W-C FIBID
    if re.search(r"W-C|W.C", s):
        return "W-C (FIBID)"
    # Bi-S based (NdOFBiS2, ...)
    if re.search(r"Bi.*S2|BiS", s, re.I):
        return "BiS2-based"
    return "Other"

dfv["family"] = dfv["material"].apply(classify_family)
print("Material families:")
print(dfv["family"].value_counts().to_string())
print(f"\nTc range per family:")
for fam, g in dfv.groupby("family"):
    tc = g["tc_best"].dropna()
    if len(tc):
        print(f"  {fam:20s}  n={len(tc):2d}   Tc = {tc.min():.1f} – {tc.max():.1f} K")

### 4a · Tc over the years

In [ ]:
# ── 4a  Tc development over the years ──
fig, ax = plt.subplots(figsize=(9, 5))

# Use numeric x-axis so we can overlay the max-Tc envelope
for fam in dfv["family"].unique():
    sub = dfv[dfv["family"] == fam]
    jitter = np.random.default_rng(hash(fam) % 2**31).uniform(-0.3, 0.3, size=len(sub))
    ax.scatter(
        sub["year"] + jitter, sub["tc_best"],
        label=fam, s=55, alpha=0.8, edgecolors="k", linewidths=0.3, zorder=3,
    )

# Per-year max Tc envelope
year_max = dfv.groupby("year")["tc_best"].max().sort_index()
ax.plot(year_max.index, year_max.values, color="k", ls="--", lw=1, alpha=0.4, zorder=1)

ax.set_xlabel("Year")
ax.set_ylabel("Tc (K)")
ax.set_title("Tc development over the years")
ax.legend(title="Family", fontsize=7, title_fontsize=8,
          bbox_to_anchor=(1.02, 1), loc="upper left", framealpha=0.9)

plt.tight_layout()
plt.show()

print(f"Year range: {dfv['year'].min()} – {dfv['year'].max()}")
print(f"Max Tc per year:")
for yr, tc in year_max.items():
    print(f"  {yr}: {tc:.1f} K")

### 4b · Material landscape — families vs Tc, coloured by synthesis method

In [ ]:
# ── 4b  Material landscape: family vs Tc, coloured by synthesis method ──
# Each dot = one material entry. Jitter along y so overlapping points separate.

# Clean up method labels
method_clean = dfv["method"].fillna("unknown").replace({"other": "other / not classified"})
method_order = (
    method_clean.value_counts()
    .index.tolist()
)

# Colour map for methods
_method_pal = {
    "solid-state":              C_BLUE,
    "arc discharge":            C_ORANGE,
    "flux growth":              C_PURPLE,
    "pulsed laser deposition":  C_PINK,
    "CVD":                      C_LBLUE,
    "other / not classified":   C_GREY,
    "unknown":                  C_DGREY,
}
# Fallback for any new methods that appear with more data
for m in method_order:
    if m not in _method_pal:
        _method_pal[m] = C_LAVENDER

# Sort families by median Tc
fam_order = (
    dfv.groupby("family")["tc_best"]
    .median()
    .sort_values()
    .index.tolist()
)

fig, ax = plt.subplots(figsize=(10, 5))

for i, fam in enumerate(fam_order):
    mask = dfv["family"] == fam
    sub = dfv[mask].copy()
    jitter = np.random.default_rng(42).uniform(-0.25, 0.25, size=len(sub))
    for _, row in sub.iterrows():
        ax.scatter(
            row["tc_best"], i + jitter[sub.index.get_loc(row.name)],
            color=_method_pal.get(method_clean[row.name], C_GREY),
            s=50, edgecolors="k", linewidths=0.3, alpha=0.8, zorder=3,
        )

ax.set_yticks(range(len(fam_order)))
ax.set_yticklabels(fam_order)
ax.set_xlabel("Tc (K)")
ax.set_title("Superconductor landscape: material family vs Tc")
ax.set_xlim(left=-2)

# Legend for methods
from matplotlib.lines import Line2D
handles = [Line2D([0], [0], marker="o", color="w", markerfacecolor=_method_pal[m],
                   markeredgecolor="k", markeredgewidth=0.3, markersize=8, label=m)
           for m in method_order if m in _method_pal]
ax.legend(handles=handles, title="Synthesis method", fontsize=7,
          title_fontsize=8, loc="lower right", framealpha=0.9)

plt.tight_layout()
plt.show()

print(f"Total points: {len(dfv)}, families: {len(fam_order)}, methods: {len(method_order)}")

### 4c · Synthesis method vs Tc — box + strip plot

In [ ]:
# ── 4c  Synthesis method vs Tc: box + strip ──
# Only rows with a known (non-unknown) synthesis method

plot_df = dfv[dfv["method"].notna()].copy()
plot_df["method_label"] = plot_df["method"].replace({"other": "other / unclassified"})

# Order by median Tc
meth_med = plot_df.groupby("method_label")["tc_best"].median().sort_values()
order = meth_med.index.tolist()

fig, ax = plt.subplots(figsize=(9, 5))

sns.boxplot(
    data=plot_df, y="method_label", x="tc_best", order=order,
    color="white", fliersize=0, linewidth=1, width=0.5, ax=ax,
)
sns.stripplot(
    data=plot_df, y="method_label", x="tc_best", order=order,
    hue="family", palette="tab10", size=7, alpha=0.8,
    edgecolor="k", linewidth=0.3, jitter=0.2, ax=ax, dodge=False,
)

ax.set_xlabel("Tc (K)")
ax.set_ylabel("")
ax.set_title("Synthesis method vs Tc (points coloured by material family)")
ax.legend(title="Family", fontsize=7, title_fontsize=8,
          bbox_to_anchor=(1.02, 1), loc="upper left", framealpha=0.9)

# Annotate counts
for i, meth in enumerate(order):
    n = (plot_df["method_label"] == meth).sum()
    ax.text(ax.get_xlim()[1] * 0.98, i, f"n={n}", va="center", ha="right", fontsize=7, color="grey")

plt.tight_layout()
plt.show()

### 4d · Extraction completeness heatmap per paper

For each paper: does the pipeline have text Tc, VLM Tc, and synthesis data?

In [ ]:
# ── 4c  Extraction completeness heatmap ──
from matplotlib.colors import LinearSegmentedColormap

cov_cols = {
    "Text Tc":        "has_text_tc",
    "VLM orig":       "has_vlm_tc_orig",
    "VLM snip":       "has_vlm_tc_snip",
    "Synthesis":      "has_synthesis",
}

# Build per-paper fraction matrix
papers = dfv.groupby("paper_id")
paper_ids = papers.size().sort_values(ascending=False).index.tolist()

mat = np.zeros((len(paper_ids), len(cov_cols)))
for i, pid in enumerate(paper_ids):
    g = dfv[dfv["paper_id"] == pid]
    for j, (label, col) in enumerate(cov_cols.items()):
        mat[i, j] = g[col].sum() / len(g)  # fraction of materials with this data

# Shorten paper IDs for display
short_ids = [pid.split("_")[0] for pid in paper_ids]

fig, ax = plt.subplots(figsize=(5, 8))
cmap = LinearSegmentedColormap.from_list("cov", ["#f0f0f0", C_BLUE])
im = ax.imshow(mat, cmap=cmap, aspect="auto", vmin=0, vmax=1)

ax.set_xticks(range(len(cov_cols)))
ax.set_xticklabels(cov_cols.keys(), rotation=45, ha="right", fontsize=9)
ax.set_yticks(range(len(paper_ids)))
ax.set_yticklabels(short_ids, fontsize=8)
ax.set_ylabel("Paper (arXiv ID)")
ax.set_title("Extraction coverage per paper")

# Annotate cells with percentage
for i in range(mat.shape[0]):
    for j in range(mat.shape[1]):
        val = mat[i, j]
        txt = f"{val:.0%}" if val > 0 else "—"
        color = "white" if val > 0.55 else "black"
        ax.text(j, i, txt, ha="center", va="center", fontsize=7, color=color)

plt.colorbar(im, ax=ax, label="Fraction of materials", shrink=0.6)
plt.tight_layout()
plt.show()

### 4e · Synthesis extraction quality vs Tc error

In [ ]:
# ── 4d  Synthesis extraction quality vs Tc accuracy ──
# Use human text Tc as ground truth where available, fall back to tc_best

# Merge human annotations if available in dfv via the CSV columns
tc_human_col = "tc_text_human" if "tc_text_human" in dfv.columns else None

# Compute Tc error: |pipeline_best - human_text| where both exist
if tc_human_col:
    dfv["_tc_human"] = pd.to_numeric(dfv[tc_human_col], errors="coerce")
    dfv["tc_error"] = (dfv["tc_best"] - dfv["_tc_human"]).abs()
else:
    dfv["tc_error"] = np.nan

plot_df = dfv.dropna(subset=["overall_score", "tc_error"]).copy()

fig, ax = plt.subplots(figsize=(6, 5))

if len(plot_df) >= 3:
    sns.scatterplot(
        data=plot_df, x="overall_score", y="tc_error",
        hue="family", palette="tab10", s=70, alpha=0.8,
        edgecolor="k", linewidth=0.3, ax=ax,
    )
    ax.set_xlabel("Synthesis extraction quality (overall score)")
    ax.set_ylabel("|Tc pipeline - Tc human text| (K)")
    ax.set_title("Synthesis quality vs Tc accuracy")
    ax.legend(title="Family", fontsize=7, title_fontsize=8,
              bbox_to_anchor=(1.02, 1), loc="upper left")
    print(f"Plotted {len(plot_df)} points with both synthesis score and Tc error")
else:
    ax.text(0.5, 0.5, f"Not enough data (n={len(plot_df)})\nNeed both overall_score and human Tc",
            transform=ax.transAxes, ha="center", va="center", fontsize=11)
    ax.set_title("Synthesis quality vs Tc accuracy — awaiting more data")
    print(f"Only {len(plot_df)} points — this plot needs more human annotations + synthesis evals")

plt.tight_layout()
plt.show()

### 4f · Synthesis method × material family — stacked bar

In [ ]:
# ── 4e  Synthesis method × material family stacked bar ──
ct = pd.crosstab(
    dfv["method"].fillna("unknown").replace({"other": "other / unclass."}),
    dfv["family"],
)
# order rows by total count
ct = ct.loc[ct.sum(axis=1).sort_values(ascending=True).index]

fig, ax = plt.subplots(figsize=(9, 5))
ct.plot.barh(stacked=True, ax=ax, colormap="tab10", edgecolor="white", linewidth=0.5)

ax.set_xlabel("Number of materials")
ax.set_ylabel("")
ax.set_title("Synthesis method vs material family")
ax.legend(title="Family", fontsize=7, title_fontsize=8,
          bbox_to_anchor=(1.02, 1), loc="upper left", framealpha=0.9)

# Count annotations
for i, (idx, row) in enumerate(ct.iterrows()):
    total = row.sum()
    ax.text(total + 0.3, i, str(int(total)), va="center", fontsize=8, color="grey")

plt.tight_layout()
plt.show()

### 4g · Max synthesis temperature vs Tc (sparse — will fill with more data)

In [ ]:
# ── 4f  Max synthesis temperature vs Tc ──
plot_df = dfv.dropna(subset=["max_temp_C", "tc_best"]).copy()

fig, ax = plt.subplots(figsize=(6, 5))

if len(plot_df) >= 3:
    sns.scatterplot(
        data=plot_df, x="max_temp_C", y="tc_best",
        hue="family", style="method", palette="tab10",
        s=80, alpha=0.85, edgecolor="k", linewidth=0.3, ax=ax,
    )
    ax.set_xlabel("Max synthesis temperature (°C)")
    ax.set_ylabel("Tc (K)")
    ax.set_title(f"Synthesis temperature vs Tc  (n={len(plot_df)})")
    ax.legend(fontsize=7, title_fontsize=8,
              bbox_to_anchor=(1.02, 1), loc="upper left")
else:
    ax.text(0.5, 0.5,
            f"Only {len(plot_df)} materials have extracted\n"
            "synthesis temperatures — needs more data",
            transform=ax.transAxes, ha="center", va="center", fontsize=11)
    ax.set_title("Synthesis temperature vs Tc — awaiting more data")

plt.tight_layout()
plt.show()
print(f"Materials with extracted synthesis temperature: {len(plot_df)}")

### 4h · Starting materials frequency (sparse — will fill with more data)

In [ ]:
# ── 4g  Starting materials frequency bar chart ──
# Explode the starting_materials list from the JSON-parsed dataframe

starting_rows = dfs[dfs["starting_materials"].apply(len) > 0].copy()

if len(starting_rows) >= 2:
    all_sm = []
    for _, row in starting_rows.iterrows():
        for sm in row["starting_materials"]:
            all_sm.append({"starting_material": sm, "method": row["method"], "material": row["material"]})
    sm_df = pd.DataFrame(all_sm)

    # Top N starting materials
    top_n = min(20, sm_df["starting_material"].nunique())
    top_sm = sm_df["starting_material"].value_counts().head(top_n)

    fig, ax = plt.subplots(figsize=(7, max(3, top_n * 0.35)))
    top_sm.sort_values().plot.barh(ax=ax, color=C_BLUE, edgecolor="white", linewidth=0.5)
    ax.set_xlabel("Occurrences across materials")
    ax.set_ylabel("")
    ax.set_title(f"Most common starting materials  (from {len(starting_rows)} materials with data)")

    plt.tight_layout()
    plt.show()

    print(f"\nTotal starting material mentions: {len(sm_df)}")
    print(f"Unique starting materials: {sm_df['starting_material'].nunique()}")
    print(f"Materials with starting material data: {len(starting_rows)}/{len(dfs)}")
else:
    print(f"Only {len(starting_rows)} materials have starting material data — skipping plot."
          " This will populate with more synthesis extractions.")

### 4i · Synthesis data sparsity summary

In [ ]:
# ── 4h  Synthesis sparsity summary ──
fields = {
    "Synthesis method":       (dfs["method"] != "unknown").sum(),
    "≥1 synthesis step":      (dfs["n_steps"] > 0).sum(),
    "≥1 starting material":   (dfs["n_starting"] > 0).sum(),
    "Has max temperature":    dfs["max_temp_C"].notna().sum(),
    "Has duration":           dfs["total_duration_h"].notna().sum(),
    "Has evaluation score":   dfs["overall_score"].notna().sum(),
}

fig, ax = plt.subplots(figsize=(7, 3.5))
labels = list(fields.keys())
counts = list(fields.values())
total = len(dfs)

bars = ax.barh(labels, counts, color=C_BLUE, edgecolor="white", linewidth=0.5)
ax.barh(labels, [total - c for c in counts], left=counts,
        color="#e8e8e8", edgecolor="white", linewidth=0.5)

for bar, c in zip(bars, counts):
    ax.text(c + 0.5, bar.get_y() + bar.get_height() / 2,
            f"{c}/{total} ({c/total:.0%})", va="center", fontsize=8)

ax.set_xlabel(f"Materials (total = {total})")
ax.set_title("Synthesis extraction completeness")
ax.set_xlim(0, total * 1.25)

plt.tight_layout()
plt.show()

print("\nPlots 4f–4g are sparse now but ready for more data."
      "\nRe-run after adding more papers to see them fill in.")